**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Optimal Transport

The geometry of *moving probability mass*: Monge's dirt-shoveling problem, Kantorovich's relaxation, the 20-line Sinkhorn algorithm (verified against exact solvers), and why Wasserstein distances fixed a real failure mode of KL in machine learning.

## 1. Pre-requisites

[Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) (KL, for the contrast), [Convex Optimization II](../Intro_Math/Optimization/Convex_Optimization_2.ipynb) (duality — OT is a linear program).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment, linprog
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Monge, Kantorovich & Why KL Fails* (~35 min)
**Goal:** transport as the distance that respects GEOMETRY; where KL divergence goes blind.
**Feeds into:** Session 2 (Sinkhorn).

---

## 2. A Distance That Knows the Ground

💡 **Intuition.** KL divergence compares distributions *pointwise*: two non-overlapping distributions are 'infinitely different' whether they're 1 mm or 1 km apart — KL never looks at the ground between them. **Optimal transport** does: $W(p, q) = $ the minimum *cost of shoveling* $p$'s mass into $q$'s shape, cost = mass × distance moved. It's finite, smooth, and its gradient says *which direction to move* — exactly what a generative model's training signal needs (the insight behind Wasserstein GANs, and the geometry under [diffusion models](./Diffusion_Models.ipynb)).

In [2]:
# two spikes sliding apart: KL slams to a wall instantly, W1 reports the distance
grid = np.linspace(0, 10, 400)
def spike(mu, w=0.15):
    p = np.exp(-(grid-mu)**2/(2*w**2)); return p/p.sum()

seps = np.linspace(0, 6, 25)
kl_vals, w1_vals = [], []
p = spike(2.0)
for s_ in seps:
    q = spike(2.0 + s_)
    m = (p > 1e-12) & (q > 1e-12)
    kl = np.sum(p[m]*np.log(p[m]/q[m])) if m.any() else np.inf
    kl_vals.append(kl)
    # W1 in 1-D has a CLOSED FORM: integral |CDF_p − CDF_q|  (our oracle for later, too)
    w1_vals.append(np.sum(np.abs(np.cumsum(p) - np.cumsum(q))) * (grid[1]-grid[0]))

fig, axes = plt.subplots(1, 2, figsize=(9, 2.8))
axes[0].plot(seps, kl_vals); axes[0].set_title("KL(p‖q): explodes, then saturates — gradient dies")
axes[1].plot(seps, w1_vals); axes[1].set_title("W₁(p, q): = the separation. informative forever")
for ax in axes: ax.set_xlabel("separation"); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"W1 slope: {np.polyfit(seps[5:], w1_vals[5:], 1)[0]:.3f}  (exactly 1: cost = distance moved)")

W1 slope: 1.000  (exactly 1: cost = distance moved)


/tmp/ipykernel_2994421/173178455.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Sinkhorn: Entropic OT in 20 Lines* (~40 min)
**Goal:** add an entropy smoothing term and OT becomes two alternating normalizations — verified twice.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (OT in ML).

---

## 3. The Algorithm That Made OT Practical

💡 **Intuition.** Exact OT is a linear program — expensive at scale. Add an entropy term $\varepsilon H(\pi)$ and the optimal plan becomes $\pi = \mathrm{diag}(u) K \mathrm{diag}(v)$ with $K = e^{-C/\varepsilon}$ — and finding $u, v$ is just **alternately normalizing rows and columns** to match the marginals. Twenty lines, all matrix-vector, GPU-loving. As $\varepsilon \to 0$ it approaches the exact plan; we verify against *two* independent oracles: the Hungarian assignment solver and a general LP.

In [3]:
def sinkhorn(a, b, C, eps=0.01, iters=2000):
    K = np.exp(-C/eps)
    u = np.ones_like(a)
    for _ in range(iters):
        v = b / (K.T @ u)
        u = a / (K @ v)
    P = u[:, None] * K * v[None, :]
    return P, np.sum(P * C)

# ORACLE 1: assignment case (uniform marginals, n=n) — Hungarian algorithm is exact
n_pts = 40
X = rng.random((n_pts, 2)); Y = rng.random((n_pts, 2)) + 0.3
C = np.sqrt(((X[:, None] - Y[None])**2).sum(-1))
a = b = np.ones(n_pts)/n_pts

row, col = linear_sum_assignment(C)
cost_exact = C[row, col].mean()
P_sk, cost_sk = sinkhorn(a, b, C, eps=0.005)
print(f"assignment oracle (Hungarian): {cost_exact:.5f}   Sinkhorn ε=0.005: {cost_sk:.5f}   gap {cost_sk-cost_exact:.2e}")

# ORACLE 2: general marginals — solve the LP directly
a2 = rng.random(15); a2 /= a2.sum()
b2 = rng.random(20); b2 /= b2.sum()
C2 = np.abs(np.linspace(0,1,15)[:, None] - np.linspace(0,1,20)[None])
A_eq = np.zeros((35, 300))
for i in range(15): A_eq[i, i*20:(i+1)*20] = 1
for j in range(20): A_eq[15+j, j::20] = 1
lp = linprog(C2.ravel(), A_eq=A_eq, b_eq=np.r_[a2, b2], bounds=(0, None), method="highs")
_, cost_sk2 = sinkhorn(a2, b2, C2, eps=0.003)
print(f"LP oracle: {lp.fun:.5f}   Sinkhorn: {cost_sk2:.5f}   gap {cost_sk2-lp.fun:.2e}")

assignment oracle (Hungarian): 0.47865   Sinkhorn ε=0.005: 0.48071   gap 2.06e-03
LP oracle: 0.06211   Sinkhorn: 0.06145   gap -6.61e-04


In [4]:
# the transport plan itself, at three smoothing levels — sharpening toward the LP vertex
fig, axes = plt.subplots(1, 3, figsize=(9.5, 2.8))
for ax, eps in zip(axes, [0.1, 0.02, 0.003]):
    P, c = sinkhorn(a2, b2, C2, eps=eps)
    ax.imshow(P, aspect="auto", cmap="viridis")
    ax.set_title(f"ε={eps}: cost {c:.4f}", fontsize=8)
plt.suptitle("entropy blurs the plan; ε→0 recovers the sharp optimal one", y=1.03)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2994421/4019669250.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *OT in Machine Learning* (~35 min)
**Goal:** Wasserstein losses, barycenters, and domain adaptation — three working miniatures.
**Builds on:** Session 2.

---

## 4. Three Jobs for a Geometric Distance

In [5]:
# (1) Wasserstein barycenter of three histograms — the *geometric* average
# vs the naive pointwise average (which invents mass in the middle)
h1, h2, h3 = spike(2.0, 0.3), spike(5.0, 0.3), spike(8.0, 0.3)
naive = (h1 + h2 + h3)/3
# 1-D W-barycenter has a quantile closed form: average the inverse CDFs (another oracle-friendly fact)
qs = np.linspace(0.001, 0.999, 400)
inv_cdfs = [np.interp(qs, np.cumsum(h), grid) for h in (h1, h2, h3)]
bary_support = np.mean(inv_cdfs, 0)
bary = np.histogram(bary_support, bins=len(grid), range=(0,10), weights=np.full(len(qs), 1/len(qs)))[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 2.4))
for h in (h1, h2, h3):
    for ax in axes: ax.plot(grid, h, "gray", linewidth=0.7, alpha=0.6)
axes[0].plot(grid, naive, "C3", linewidth=1.5); axes[0].set_title("pointwise mean: three ghosts")
axes[1].plot(grid, bary, "C0", linewidth=1.5); axes[1].set_title("Wasserstein barycenter: ONE shape, centered")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2994421/368051850.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# (2) domain adaptation: transport source samples onto the target cloud, then classify
Xs = rng.standard_normal((150, 2)) * 0.5 + [0, 0]
ys = (Xs[:, 0] > 0).astype(int)
theta = 0.9
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
Xt = (rng.standard_normal((150, 2)) * 0.5) @ R.T + [2.5, 1.0]      # rotated + shifted domain
yt = ((Xt - [2.5, 1.0]) @ R)[:, 0] > 0                              # true labels in target frame

C_da = ((Xs[:, None] - Xt[None])**2).sum(-1)
P_da, _ = sinkhorn(np.ones(150)/150, np.ones(150)/150, C_da, eps=0.05)
Xs_mapped = (P_da / P_da.sum(1, keepdims=True)) @ Xt               # barycentric mapping

from numpy.linalg import lstsq
w_naive = lstsq(np.c_[Xs, np.ones(150)], ys*2-1, rcond=None)[0]
w_ot    = lstsq(np.c_[Xs_mapped, np.ones(150)], ys*2-1, rcond=None)[0]
acc = lambda w: np.mean(((np.c_[Xt, np.ones(150)] @ w) > 0) == yt)
print(f"classify the TARGET domain — train on source: {acc(w_naive):.0%}   train on OT-mapped source: {acc(w_ot):.0%}")

classify the TARGET domain — train on source: 49%   train on OT-mapped source: 73%


**(3) And the one you've already met:** [diffusion models](./Diffusion_Models.ipynb) learn a path between noise and data; the probability-flow view of that path is a transport map, and 'flow matching' — the current frontier — trains it with explicitly OT-inspired straight-line couplings. The shoveling metaphor became the state of the art.

## 5. Conclusion

OT measures distribution distance *through the ground metric* (where KL is blind), Sinkhorn computes it at scale (verified against Hungarian and LP oracles to 1e-3), and barycenters/adaptation/flows cash it in ML. Geometry, not pointwise comparison.

---
## Where next

- [Diffusion Models](./Diffusion_Models.ipynb) & the score/flow frontier.
- [Convex Optimization II](../Intro_Math/Optimization/Convex_Optimization_2.ipynb) — OT's duality (Kantorovich–Rubinstein) is a beautiful exercise.